In [1]:
import xml.etree.ElementTree as ET
import pandas as pd 
from datetime import datetime, timedelta
from io import StringIO

## Test Code

In [6]:
filename = "../data/predictions/02_23_26_07:00-stop-256.xml"

In [7]:
text = pd.read_xml(filename)
text

,agencyTitle,routeTitle,routeTag,stopTitle,stopTag,direction,dirTitleBecauseNoPredictions
0,Unitrans ASUCD/City of Davis,C,C,Silo Terminal & Center Island (WB),22256,\n,None
1,Unitrans ASUCD/City of Davis,Vl,VL,Silo Terminal & Center Island (WB),22256,\n,None
2,Unitrans ASUCD/City of Davis,V,V,Silo Terminal & Center Island (WB),22256,None,Outbound to West Village (all Stops)
3,Unitrans ASUCD/City of Davis,Vx,VX,Silo Terminal & Center Island (WB),22256,\n,None
4,Unitrans ASUCD/City of Davis,J,J,Silo Terminal & Center Island (WB),22256,\n,None
5,Unitrans ASUCD/City of Davis,D,D,Silo Terminal & Center Island (WB),22256,\n,None


In [8]:
# check for available data
try:
    text_layer1 = pd.read_xml(filename, xpath=".//direction")
    print(text_layer1)

except Exception as e:
    print(f"No direction information found for route/time combination: {e}\nskipping...")


                                               title  prediction
0                            Outbound to Wake Forest         NaN
1  Outbound to West Village - No Service to The G...         NaN
2               Outbound to The Green Only (express)         NaN
3                             Outbound to N Sycamore         NaN
4                              Outbound to Arlington         NaN


In [9]:
tree = ET.parse(filename)
root = tree.getroot()

prediction_map = {prediction.get("routeTag"): prediction.text for prediction in root.findall(".//predictions")}
print(prediction_map)


text_layer2 = pd.read_xml(filename, xpath=".//prediction")
text_layer2

{'C': '\n  ', 'VL': '\n  ', 'V': '\n', 'VX': '\n  ', 'J': '\n  ', 'D': '\n  '}


,epochTime,seconds,minutes,isDeparture,affectedByLayover,dirTag,vehicle,block,tripTag
0,1771860300000,1495,24,True,True,C_0_var0,LMU_4096,7,C_11_outbound_0725
1,1771863900000,5095,84,True,True,C_0_var0,LMU_4096,7,C_11_outbound_0825
2,1771860300000,1495,24,True,True,VL_0_var0,LMU_8114,10,VL_11_outbound_0725
3,1771862100000,3295,54,True,True,VL_0_var0,LMU_8114,10,VL_11_outbound_0755
4,1771863900000,5095,84,True,True,VL_0_var0,LMU_8114,10,VL_11_outbound_0825
5,1771860300000,1495,24,True,True,VX_0_var0,LMU_8113,9,VX_11_outbound_0725
6,1771862100000,3295,54,True,True,VX_0_var0,LMU_8113,9,VX_11_outbound_0755
7,1771863900000,5095,84,True,True,VX_0_var0,LMU_8113,9,VX_11_outbound_0825
8,1771859700000,895,14,True,True,J_0_var0,LMU_4021,4,J_11_outbound_0715
9,1771860900000,2095,34,True,True,J_0_var0,LMU_4091,3,J_11_outbound_0735


In [10]:
rows = []
for prediction in root.findall(".//predictions"):
    route_title = prediction.get('routeTitle')
    route_tag = prediction.get('routeTag')
    stop_title = prediction.get('stopTitle')
    stop_tag = prediction.get('stopTag')
    direction = prediction.find("direction")

    if direction is not None:
        pred = direction.find('prediction')

        if pred is not None:
            rows.append({
                'route_title': route_title,
                'route_tag': route_tag,
                'stop_title': stop_title,
                'stop_tag': stop_tag,
                'direction': direction.get('title'),
                'mins': pred.get('minutes'),
                'secs': pred.get('seconds'),
                'epoch': pred.get('epochTime'),
                'is_departure': pred.get('isDeparture'),
                'affected_by_layover': pred.get('affectedByLayover'),
                'vehicle': pred.get('vehicle'),
                'block' : pred.get('block'),
                'trip_tag': pred.get('tripTag')
            })

pd.DataFrame(rows)

,route_title,route_tag,stop_title,stop_tag,direction,mins,secs,epoch,is_departure,affected_by_layover,vehicle,block,trip_tag
0,C,C,Silo Terminal & Center Island (WB),22256,Outbound to Wake Forest,24,1495,1771860300000,true,true,LMU_4096,7,C_11_outbound_0725
1,Vl,VL,Silo Terminal & Center Island (WB),22256,Outbound to West Village - No Service to The G...,24,1495,1771860300000,true,true,LMU_8114,10,VL_11_outbound_0725
2,Vx,VX,Silo Terminal & Center Island (WB),22256,Outbound to The Green Only (express),24,1495,1771860300000,true,true,LMU_8113,9,VX_11_outbound_0725
3,J,J,Silo Terminal & Center Island (WB),22256,Outbound to N Sycamore,14,895,1771859700000,true,true,LMU_4021,4,J_11_outbound_0715
4,D,D,Silo Terminal & Center Island (WB),22256,Outbound to Arlington,54,3295,1771862100000,true,true,LMU_4096,7,D_11_outbound_0755


## Prediction Parsing Method

In [11]:
def parse_unitrans_predictions(file_path:str, base_time: datetime) -> pd.DataFrame:
    """
    Takes in the file path and the parsed base time from the filename
    Returns a df of the parsed xml file with additional features
    """

    # inital setup
    tree = ET.parse(file_path)
    root = tree.getroot()

    rows = []
    # loop through each prediction block
    for prediction in root.findall(".//predictions"):
        # get the header attributes
        route_title = prediction.get('routeTitle')
        route_tag = prediction.get('routeTag')
        stop_title = prediction.get('stopTitle')
        stop_tag = prediction.get('stopTag')
        direction = prediction.find("direction")

        # check the there is a valid direction
        if direction is not None:
            # if there is, grab the first one
            pred = direction.find('prediction')

            if pred is not None:
                # get the mins until arrival
                mins_val = int(pred.get('minutes', 0))

                # use the base time from the filename to calculate the arrival time
                predicted_arrival = base_time + timedelta(minutes=mins_val)
                
                # append all the features 
                rows.append({
                    'route_title': route_title,
                    'route_tag': route_tag,
                    'stop_title': stop_title,
                    'stop_tag': stop_tag,
                    'file_time': base_time,
                    'predicted_arrival_at': predicted_arrival,
                    'direction': direction.get('title').lower(),
                    'mins': pred.get('minutes'),
                    'secs': pred.get('seconds'),
                    'is_departure': pred.get('isDeparture'),
                    'affected_by_layover': pred.get('affectedByLayover'),
                    'vehicle': pred.get('vehicle'),
                    'block' : pred.get('block'),
                    'trip_tag': pred.get('tripTag')
                })

    df = pd.DataFrame(rows)
    if not df.empty:
        # set the arrival time to a datetime object
        df['predicted_arrival_at'] = pd.to_datetime(df['predicted_arrival_at'])
        
        # get the day of the week from the predicted arrival time
        df['day_name'] = df['predicted_arrival_at'].dt.day_name()

        # use the day name to assign the service class
        df['service_class'] = df['day_name'].apply(
            lambda day: 'MoTuTh' if day in ['Monday', 'Tuesday', 'Thursday'] else day
        )

        # ensure block is a string - for merging
        df['block'] = df['block'].astype(str)
    return df

In [12]:
import os

In [13]:
def extract_time_from_filename(filename):
    # extract the datetime from the filename
    base = os.path.basename(filename).split('-stop-')[0]
    return datetime.strptime(base, "%m_%d_%y_%H:%M")

In [14]:
# define all folders to process in one list
# list comprehension for the Feb dates + adding March 1st
folders = [f'../data/predictions/02_{i:02d}_26' for i in range(23, 29)] + ['../data/predictions/03_01_26']

all_dfs = []

for folder in folders:
    if not os.path.exists(folder):
        continue
        
    for file in os.listdir(folder):
        file_path = os.path.join(folder, file)
        
        try:
            file_dt = extract_time_from_filename(file)
            # ensure the xml is valid
            pd.read_xml(file_path, xpath=".//direction") 
            
            # parse and store the dataframe
            current_df = parse_unitrans_predictions(file_path=file_path, base_time=file_dt)
            all_dfs.append(current_df)
            
        except Exception:
            continue

# concatenate everything at once 
master_df = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()

In [15]:
master_df

,route_title,route_tag,stop_title,stop_tag,file_time,predicted_arrival_at,direction,mins,secs,is_departure,affected_by_layover,vehicle,block,trip_tag,day_name,service_class
0,Z,Z,Silo Terminal & Haring Hall (WB),22258,2026-02-24 06:00:00,2026-02-24 06:24:00,outbound to target,24,1494,true,true,LMU_4096,46,Z_11_outbound_0625,Tuesday,MoTuTh
1,W,W,Silo Terminal & Haring Hall (WB),22258,2026-02-24 06:10:00,2026-02-24 07:34:00,outbound to drummond,84,5094,true,true,LMU_4010,1,W_11_outbound_0735,Tuesday,MoTuTh
2,Q,Q,Memorial Union & Main Island (NB),22272,2026-02-24 06:10:00,2026-02-24 06:24:00,clockwise to ucd mu,14,894,true,true,LMU_4084,58,Q_11_0625,Tuesday,MoTuTh
3,L,L,Silo Terminal & Haring Hall (WB),22258,2026-02-24 06:15:00,2026-02-24 06:24:00,outbound to loyola,9,595,true,true,LMU_4094,45,L_11_outbound_0625,Tuesday,MoTuTh
4,W,W,Silo Terminal & Haring Hall (WB),22258,2026-02-24 06:15:00,2026-02-24 07:34:00,outbound to drummond,79,4795,true,true,LMU_4010,1,W_11_outbound_0735,Tuesday,MoTuTh
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13491,U,U,Memorial Union & East Island (SB),22274,2026-03-01 18:25:00,2026-03-01 18:34:00,outbound to west village via russell blvd,9,596,true,true,LMU_4011,65,U_20_outbound_1835,Sunday,Sunday
13492,M,M,Memorial Union & Main Island (SB),22273,2026-03-01 18:30:00,2026-03-01 18:34:00,outbound to drew,4,296,true,true,LMU_4020,64,M_20_outbound_1835,Sunday,Sunday
13493,U,U,Memorial Union & East Island (SB),22274,2026-03-01 18:30:00,2026-03-01 18:35:00,outbound to west village via russell blvd,5,339,true,true,LMU_4011,65,U_20_outbound_1835,Sunday,Sunday
13494,M,M,Memorial Union & Main Island (SB),22273,2026-03-01 18:35:00,2026-03-01 18:35:00,outbound to drew,0,40,true,true,LMU_4020,64,M_20_outbound_1835,Sunday,Sunday


In [16]:
import sqlite3

In [17]:
# connect to existing database
conn = sqlite3.connect('unitrans.db')

# create predictions table
master_df.to_sql('predictions', conn, if_exists='replace', index=False)

pd.read_sql_query("SELECT * FROM predictions", conn)

,route_title,route_tag,stop_title,stop_tag,file_time,predicted_arrival_at,direction,mins,secs,is_departure,affected_by_layover,vehicle,block,trip_tag,day_name,service_class
0,Z,Z,Silo Terminal & Haring Hall (WB),22258,2026-02-24 06:00:00,2026-02-24 06:24:00,outbound to target,24,1494,true,true,LMU_4096,46,Z_11_outbound_0625,Tuesday,MoTuTh
1,W,W,Silo Terminal & Haring Hall (WB),22258,2026-02-24 06:10:00,2026-02-24 07:34:00,outbound to drummond,84,5094,true,true,LMU_4010,1,W_11_outbound_0735,Tuesday,MoTuTh
2,Q,Q,Memorial Union & Main Island (NB),22272,2026-02-24 06:10:00,2026-02-24 06:24:00,clockwise to ucd mu,14,894,true,true,LMU_4084,58,Q_11_0625,Tuesday,MoTuTh
3,L,L,Silo Terminal & Haring Hall (WB),22258,2026-02-24 06:15:00,2026-02-24 06:24:00,outbound to loyola,9,595,true,true,LMU_4094,45,L_11_outbound_0625,Tuesday,MoTuTh
4,W,W,Silo Terminal & Haring Hall (WB),22258,2026-02-24 06:15:00,2026-02-24 07:34:00,outbound to drummond,79,4795,true,true,LMU_4010,1,W_11_outbound_0735,Tuesday,MoTuTh
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13491,U,U,Memorial Union & East Island (SB),22274,2026-03-01 18:25:00,2026-03-01 18:34:00,outbound to west village via russell blvd,9,596,true,true,LMU_4011,65,U_20_outbound_1835,Sunday,Sunday
13492,M,M,Memorial Union & Main Island (SB),22273,2026-03-01 18:30:00,2026-03-01 18:34:00,outbound to drew,4,296,true,true,LMU_4020,64,M_20_outbound_1835,Sunday,Sunday
13493,U,U,Memorial Union & East Island (SB),22274,2026-03-01 18:30:00,2026-03-01 18:35:00,outbound to west village via russell blvd,5,339,true,true,LMU_4011,65,U_20_outbound_1835,Sunday,Sunday
13494,M,M,Memorial Union & Main Island (SB),22273,2026-03-01 18:35:00,2026-03-01 18:35:00,outbound to drew,0,40,true,true,LMU_4020,64,M_20_outbound_1835,Sunday,Sunday
